In [1]:
# 02 Preprocessing & Feature Engineering
# Projet: Telemed Urgence IA

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import os

# style global
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✅ Imports OK")

✅ Imports OK


In [2]:
# 02Chargement des données

df = pd.read_csv("../data/raw/dataset_telemed.csv")

# Suppression de patient_id (identifiant direct, inutile pour la modélisation)
# C'est une obligation RGPD: on ne garde pas les identifiants directs
df = df.drop(columns=['patient_id'])

# séparation features/cible
X = df.drop(columns=['niveau_urgence'])
y = df['niveau_urgence']

print(f"✅ Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"✅ Features : {X.shape[1]} colonnes")
print(f"✅ Cible : {y.shape[0]} valeurs")
print(f"\n📊 Distribution cible :")
print(y.value_counts())

✅ Dataset chargé : 10080 lignes × 12 colonnes
✅ Features : 11 colonnes
✅ Cible : 10080 valeurs

📊 Distribution cible :
niveau_urgence
0    5053
1    3551
2    1476
Name: count, dtype: int64


In [3]:
# 03 Définition des colonnes par type

# Colonne texte sera vectorisée séparément avec TF-IDF
col_text = 'description_symptomes'

# Variables numériques =>imputation médiane + standardisation
col_num = [
    'age', 
    'freq_cardiaque', 
    'tension_sys', 
    'temp', 
    'sat_oxygene', 
    'antecedents',
    'duree_symptomes'
]

# Variables catégorielles => imputation mode + One-Hot Encoding
col_cat = [
    'sexe', 
    'zone_vie', 
    'source'
]

print("✅ Colonnes définies :")
print(f"   • Numériques  ({len(col_num)}) : {col_num}")
print(f"   • Catégorielles ({len(col_cat)}) : {col_cat}")
print(f"   • Texte        (1) : {col_text}")

✅ Colonnes définies :
   • Numériques  (7) : ['age', 'freq_cardiaque', 'tension_sys', 'temp', 'sat_oxygene', 'antecedents', 'duree_symptomes']
   • Catégorielles (3) : ['sexe', 'zone_vie', 'source']
   • Texte        (1) : description_symptomes


In [4]:
# 04 Split Train/Test
# On fixe random_state=42 pour la reproductibilité
# stratify = y garantit que les 3 classes sont proportionnellement
# représentées dans le train & le test (crucial avec déséquilibre)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✅ Split effectué :")
print(f"   • Train : {X_train.shape[0]} échantillons ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   • Test  : {X_test.shape[0]} échantillons ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\n📊 Distribution classes — Train :")
print(y_train.value_counts().sort_index())

print(f"\n📊 Distribution classes — Test :")
print(y_test.value_counts().sort_index())

✅ Split effectué :
   • Train : 8064 échantillons (80.0%)
   • Test  : 2016 échantillons (20.0%)

📊 Distribution classes — Train :
niveau_urgence
0    4042
1    2841
2    1181
Name: count, dtype: int64

📊 Distribution classes — Test :
niveau_urgence
0    1011
1     710
2     295
Name: count, dtype: int64


In [5]:
# 05 Pipelines de preprocessing:

# Pipeline numérique
# Étape 1: Imputation par la médiane (robuste aux outliers)
# Étape 2: Standardisation (moyenne = 0, écart-type= 1)
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline catégoriel
# Étape 1 : Imputation par le mode (valeur la plus fréquente)
# Étape 2 : One-Hot Encoding (handle_unknown='ignore' pour les valeurs inconnues en prod)
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline texte
# TF-IDF: transforme le texte en vecteur numérique
# max_features=3000 : on garde les 3000 termes les plus discriminants
# ngram_range=(1,2) : unigrammes ET bigrammes (ex: "douleur thoracique")
# sublinear_tf=True : atténue l'effet des mots très fréquents
text_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('tfidf', TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        sublinear_tf=True
    ))
])

print("✅ Pipelines définis :")
print("   • Numérique  : Imputation médiane → StandardScaler")
print("   • Catégoriel : Imputation mode → OneHotEncoder")
print("   • Texte      : Imputation vide → TF-IDF (3000 features, bigrammes)")

✅ Pipelines définis :
   • Numérique  : Imputation médiane → StandardScaler
   • Catégoriel : Imputation mode → OneHotEncoder
   • Texte      : Imputation vide → TF-IDF (3000 features, bigrammes)


In [8]:
# 06 ColumnTransformer: assemblage des pipelines (corrigé)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin

# Transformer custom pour gérer le texte proprement
# (convertit la colonne en liste de strings avant TF-IDF)
class TextSelector(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        # Remplace les NaN par chaîne vide et retourne une liste
        if hasattr(X, 'iloc'):
            return X.iloc[:, 0].fillna('').tolist()
        return X.fillna('').tolist()

# pipeline texte corrigé (sans SimpleImputer)
text_pipeline = Pipeline(steps=[
    ('selector', TextSelector()),
    ('tfidf', TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        sublinear_tf=True
    ))
])

# ColumnTransformer reassemblé
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, col_num),
    ('cat', categorical_pipeline, col_cat),
    ('text', text_pipeline, [col_text])  # liste avec crochets!
], remainder='drop')

print("✅ ColumnTransformer corrigé et prêt !")

✅ ColumnTransformer corrigé et prêt !


In [ ]:
# 05-06-07 - Preprocessing complet (version corrigée)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

# Colonnes
col_num = ['age', 'freq_cardiaque', 'tension_sys', 'temp',
           'sat_oxygene', 'antecedents', 'duree_symptomes']
col_cat = ['sexe', 'zone_vie', 'source']
col_text = 'description_symptomes'

# Pipelines tabulaires
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# TF-IDF séparé (pas dans ColumnTransformer)
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# ColumnTransformer uniquement sur tabulaire
tabular_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, col_num),
    ('cat', categorical_pipeline, col_cat),
], remainder='drop')

# Fit sur train uniquement
print("Fit en cours...")

# Tabulaire
X_train_tab = tabular_preprocessor.fit_transform(X_train)
X_test_tab  = tabular_preprocessor.transform(X_test)

# Texte
train_text = X_train[col_text].fillna('').tolist()
test_text  = X_test[col_text].fillna('').tolist()
X_train_text = tfidf.fit_transform(train_text)
X_test_text  = tfidf.transform(test_text)

# Concaténation tabulaire + texte
X_train_processed = sp.hstack([X_train_tab, X_train_text])
X_test_processed  = sp.hstack([X_test_tab,  X_test_text])

print(f"✅ Preprocessing terminé !")
print(f"\n📊 Dimensions après transformation :")
print(f"   • X_train : {X_train_processed.shape}")
print(f"   • X_test  : {X_test_processed.shape}")
print(f"\n💡 Détail features :")
print(f"   • Tabulaire : {X_train_tab.shape[1]} features")
print(f"   • TF-IDF    : {X_train_text.shape[1]} features")
print(f"   • Total     : {X_train_processed.shape[1]} features")

⏳ Fit en cours...
✅ Preprocessing terminé !

📊 Dimensions après transformation :
   • X_train : (8064, 583)
   • X_test  : (2016, 583)

💡 Détail features :
   • Tabulaire : 13 features
   • TF-IDF    : 570 features
   • Total     : 583 features


In [10]:
# 08 Sauvegarde du preprocessor et des données

import joblib
import os

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../src/models", exist_ok=True)

# save du preprocessor (tabulaire + tfidf)
joblib.dump(tabular_preprocessor, '../src/models/tabular_preprocessor.joblib')
joblib.dump(tfidf, '../src/models/tfidf_vectorizer.joblib')

# Sauvegarde des splits pour réutilisation dans le notebook modélisation
joblib.dump((X_train, X_test, y_train, y_test), '../data/processed/train_test_split.joblib')
joblib.dump((X_train_processed, X_test_processed, y_train, y_test), '../data/processed/train_test_processed.joblib')

print("✅ Sauvegardés :")
print("   • src/models/tabular_preprocessor.joblib")
print("   • src/models/tfidf_vectorizer.joblib")
print("   • data/processed/train_test_split.joblib")
print("   • data/processed/train_test_processed.joblib")

✅ Sauvegardés :
   • src/models/tabular_preprocessor.joblib
   • src/models/tfidf_vectorizer.joblib
   • data/processed/train_test_split.joblib
   • data/processed/train_test_processed.joblib


In [11]:
# 09 Préparation des 4 scénarios

# Scénario 1 : Multimodal complet (déjà fait)
X_train_s1 = X_train_processed
X_test_s1  = X_test_processed
print(f"✅ S1 — Multimodal complet     : {X_train_s1.shape[1]} features")

# scénario 2 : Sans variables sensibles
# On retire : sexe, zone_vie, antecedents
# Justification RGPD : variables discriminantes sans apport clinique prouvé
col_num_s2 = ['age', 'freq_cardiaque', 'tension_sys', 'temp',
              'sat_oxygene', 'duree_symptomes']  # sans antecedents
col_cat_s2 = ['source']  # sans sexe et zone_vie

num_pipe_s2 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe_s2 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
tab_prep_s2 = ColumnTransformer(transformers=[
    ('num', num_pipe_s2, col_num_s2),
    ('cat', cat_pipe_s2, col_cat_s2),
], remainder='drop')

tfidf_s2 = TfidfVectorizer(max_features=3000, ngram_range=(1,2), sublinear_tf=True)

X_train_tab_s2 = tab_prep_s2.fit_transform(X_train)
X_test_tab_s2  = tab_prep_s2.transform(X_test)
X_train_text_s2 = tfidf_s2.fit_transform(X_train[col_text].fillna('').tolist())
X_test_text_s2  = tfidf_s2.transform(X_test[col_text].fillna('').tolist())

X_train_s2 = sp.hstack([X_train_tab_s2, X_train_text_s2])
X_test_s2  = sp.hstack([X_test_tab_s2,  X_test_text_s2])
print(f"✅ S2 — Sans variables sensibles : {X_train_s2.shape[1]} features")

# Scénario 3 : NLP seul
# Uniquement description_symptomes vectorisé en TF-IDF
tfidf_s3 = TfidfVectorizer(max_features=3000, ngram_range=(1,2), sublinear_tf=True)
X_train_s3 = tfidf_s3.fit_transform(X_train[col_text].fillna('').tolist())
X_test_s3  = tfidf_s3.transform(X_test[col_text].fillna('').tolist())
print(f"✅ S3 — NLP seul               : {X_train_s3.shape[1]} features")

# Scénario 4 : Tabulaire seul
# Uniquement constantes vitales + age, sans texte
col_num_s4 = ['age', 'freq_cardiaque', 'tension_sys', 'temp', 'sat_oxygene']

num_pipe_s4 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
tab_prep_s4 = ColumnTransformer(transformers=[
    ('num', num_pipe_s4, col_num_s4),
], remainder='drop')

X_train_s4 = tab_prep_s4.fit_transform(X_train)
X_test_s4  = tab_prep_s4.transform(X_test)
print(f"✅ S4 — Tabulaire seul         : {X_train_s4.shape[1]} features")

print(f"\n🎯 Les 4 scénarios sont prêts !")

✅ S1 — Multimodal complet     : 583 features
✅ S2 — Sans variables sensibles : 578 features
✅ S3 — NLP seul               : 570 features
✅ S4 — Tabulaire seul         : 5 features

🎯 Les 4 scénarios sont prêts !


In [12]:
# 10 - Sauvegarde des 4 scénarios
# dictionnaire des scénarios pour itérer facilement dans le notebook modélisation
scenarios = {
    'S1_multimodal': {
        'X_train': X_train_s1,
        'X_test' : X_test_s1,
        'description': 'Multimodal complet (tabulaire + texte)'
    },
    'S2_sans_sensibles': {
        'X_train': X_train_s2,
        'X_test' : X_test_s2,
        'description': 'Sans variables sensibles (sexe, zone_vie, antecedents)'
    },
    'S3_nlp_seul': {
        'X_train': X_train_s3,
        'X_test' : X_test_s3,
        'description': 'NLP seul (description_symptomes uniquement)'
    },
    'S4_tabulaire_seul': {
        'X_train': X_train_s4,
        'X_test' : X_test_s4,
        'description': 'Tabulaire seul (constantes vitales + age)'
    },
}

# sauvegarde
joblib.dump(scenarios, '../data/processed/scenarios.joblib')
joblib.dump((y_train, y_test), '../data/processed/labels.joblib')

print("✅ Scénarios sauvegardés :")
for name, s in scenarios.items():
    print(f"   • {name:25s} : {s['X_train'].shape[1]} features — {s['description']}")

print(f"\n✅ Labels sauvegardés : y_train ({len(y_train)}) / y_test ({len(y_test)})")
print(f"\n🚀 Preprocessing terminé — Prêt pour la modélisation !")

✅ Scénarios sauvegardés :
   • S1_multimodal             : 583 features — Multimodal complet (tabulaire + texte)
   • S2_sans_sensibles         : 578 features — Sans variables sensibles (sexe, zone_vie, antecedents)
   • S3_nlp_seul               : 570 features — NLP seul (description_symptomes uniquement)
   • S4_tabulaire_seul         : 5 features — Tabulaire seul (constantes vitales + age)

✅ Labels sauvegardés : y_train (8064) / y_test (2016)

🚀 Preprocessing terminé — Prêt pour la modélisation !
